In [1]:
# Load and save global PM2.5 mortality in single files
# Calculate total mortality for all health variables and collate all years

In [2]:
import os
import glob
import xarray as xr
from utils.utils import get_scenario_config

In [3]:
# Number of samples: determines full distribution or stats
n_samples = 200

In [4]:
# === Health variables ===
# COPD, DIABETES, ISCHEMIC_HEART_DISEASE, LOWER_RESPIRATORY_INFECTIONS, LUNG_CANCER, STROKE
# resp_copd, t2_dm, cvd_ihd, lri, neo_lung, cvd_stroke
health_vars = ["COPD", "DIABETES", "ISCHEMIC_HEART_DISEASE",
               "LOWER_RESPIRATORY_INFECTIONS", "LUNG_CANCER", "STROKE"]

In [8]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245_G6"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]
dates = f"{years.start}-{years.stop}"

MORT_DIR = f"/glade/work/awells/air_quality/{model}/mortality/pm25/global/{n_samples}_samples/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/mortality/pm25/"

for health_VAR in health_vars:
    print(f"Processing health variable {health_VAR}")
    for ens_num in ensemble_members:
        print(f"Processing ensemble number {ens_num:02d}")

        # === FULL DIST. OR JUST STATS ===
        if n_samples <= 500:
            files = f"Global_mortality_{health_VAR}_{n_samples}samples_{model}_{scenario}_{ens_num:02d}_*.nc"
            description = (f"Global {health_VAR} mortality due to PM2.5 "
                           " - scripts by A.F. Wells (2025)")
            out_file = f"Global_mortality_{health_VAR}_{n_samples}samples_{model}_{scenario}_{ens_num:02d}_{dates}.nc"

        else:
            files = f"Global_mortality_stats_{health_VAR}_{model}_{scenario}_{ens_num:02d}_*.nc"
            description = (f"Global {health_VAR} mortality due to PM2.5 "
                           "statistics: including mean, median, "
                           "and the 95% CI - scripts by A.F. Wells (2025)")
            out_file = f"Global_mortality_stats_{health_VAR}_{model}_{scenario}_{ens_num:02d}_{dates}.nc"

        file_path = os.path.join(MORT_DIR, files)

        ds = xr.open_mfdataset(
            sorted(glob.glob(file_path)),
            combine="nested",
            concat_dim="year")
        ds = ds.assign_coords(year=range(years.start, years.stop + 1))  # range for "years" misses final year

        ds.attrs["description"] = description
        ds.attrs["model"] = model
        ds.attrs["scenario"] = scenario
        ds.attrs["ensemble_number"] = ens_num

        out_path = os.path.join(SAVE_DIR, out_file)
        print(f"Saving {out_path}")
        ds.to_netcdf(out_path)

print("All processing complete.")

Processing health variable COPD
Processing ensemble number 01
Saving /glade/work/awells/air_quality/CESM2/mortality/pm25/Global_mortality_COPD_200samples_CESM2_SSP245_G6_01_2020-2084.nc
Processing ensemble number 02
Saving /glade/work/awells/air_quality/CESM2/mortality/pm25/Global_mortality_COPD_200samples_CESM2_SSP245_G6_02_2020-2084.nc
Processing ensemble number 03
Saving /glade/work/awells/air_quality/CESM2/mortality/pm25/Global_mortality_COPD_200samples_CESM2_SSP245_G6_03_2020-2084.nc
Processing health variable DIABETES
Processing ensemble number 01
Saving /glade/work/awells/air_quality/CESM2/mortality/pm25/Global_mortality_DIABETES_200samples_CESM2_SSP245_G6_01_2020-2084.nc
Processing ensemble number 02
Saving /glade/work/awells/air_quality/CESM2/mortality/pm25/Global_mortality_DIABETES_200samples_CESM2_SSP245_G6_02_2020-2084.nc
Processing ensemble number 03
Saving /glade/work/awells/air_quality/CESM2/mortality/pm25/Global_mortality_DIABETES_200samples_CESM2_SSP245_G6_03_2020-2084

In [9]:
# Save the sum of all mortality outcomes

for ens_num in ensemble_members:
    # Find all files for this ensemble

    # === FULL DIST. OR JUST STATS ===
    if n_samples <= 500:
        in_files = f"Global_mortality_*_{n_samples}samples_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
        description = ("Total global mortality due to PM2.5 "
                       "- scripts by A.F. Wells (2025)")
    else:
        in_files = f"Global_mortality_stats_*_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
        description = ("Total global mortality due to PM2.5 "
                       "statistics: including mean, median, "
                       "and the 95% CI - scripts by A.F. Wells (2025)")
    in_path = os.path.join(SAVE_DIR, in_files)
    files = sorted(glob.glob(in_path))

    # Open and combine
    datasets = [xr.open_dataarray(f) for f in files]

    # Align (important in case of slight coordinate mismatches)
    aligned = xr.align(*datasets, join="exact")

    # Sum across the health variables
    summed_da = sum(aligned)

    summed_da.attrs["description"] = description
    summed_da.attrs["ensemble_number"] = ens_num
    summed_da.attrs["scenario"] = scenario
    summed_da.attrs["model"] = model

    out_file = f"Global_mortality_{n_samples}samples_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    out_path = os.path.join(SAVE_DIR, out_file)

    print(f"Saving summed mortality timeseries to {out_path}")
    summed_da.to_netcdf(out_path)

print("All processing complete.")

Saving summed mortality timeseries to /glade/work/awells/air_quality/CESM2/mortality/pm25/Global_mortality_200samples_CESM2_SSP245_G6_01_2020-2084.nc
Saving summed mortality timeseries to /glade/work/awells/air_quality/CESM2/mortality/pm25/Global_mortality_200samples_CESM2_SSP245_G6_02_2020-2084.nc
Saving summed mortality timeseries to /glade/work/awells/air_quality/CESM2/mortality/pm25/Global_mortality_200samples_CESM2_SSP245_G6_03_2020-2084.nc
All processing complete.
